# EDA 14: Reactome, STRING, BioGRID, and the Explanation Graph

## Purpose

Reactome describes biological pathways, STRING describes functional associations, and BioGRID records experimentally curated interactions. These sources can explain why two genes might be connected. They do not prove synthetic lethality and must remain visible evidence rather than a hidden score.

## Column dictionary

| Source / fields | Meaning | Project use |
| --- | --- | --- |
| Reactome `gene_id`, `pathway_id`, `pathway_name`, `evidence_code` | Gene membership in a named biological process. | Shared-pathway explanation. |
| Reactome parent/child pathway IDs | Pathway hierarchy. | Shows broad and specific pathway relationships. |
| STRING `gene_a_id`, `gene_b_id`, `combined_score` | Functional association and confidence score; retained edges are >=700. | High-confidence association evidence, not necessarily physical binding. |
| BioGRID gene IDs, `experimental_system`, `n_curated_records` | Curated interaction and experiment type. | Displays what kind of experiment supports the connection. |
| Graph node IDs/types/labels | Genes and pathways. | Dashboard entities. |
| Graph source/target, relation, source dataset, evidence | Typed source-preserving connection. | Dashboard explanation graph. |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown,display
pd.set_option('display.max_columns',100); pd.set_option('display.max_rows',100); sns.set_theme(style='whitegrid')
def find_root(s):
    for c in (s,*s.parents):
        if (c/'data'/'depmap_24Q4'/'architecture'/'model_master.csv').exists(): return c
    raise FileNotFoundError('Run inside az-team25.')
ROOT=find_root(Path.cwd().resolve()); DATA=ROOT/'data'; CORE=DATA/'depmap_24Q4'; EXT=DATA/'external'
master=pd.read_csv(CORE/'architecture'/'model_master.csv',low_memory=False)

react=pd.read_csv(EXT/'reactome'/'derived'/'reactome_gene_pathway.csv',low_memory=False); pathways=pd.read_csv(EXT/'reactome'/'derived'/'reactome_pathways.csv'); hierarchy=pd.read_csv(EXT/'reactome'/'derived'/'reactome_pathway_hierarchy.csv'); string=pd.read_csv(EXT/'string_v12'/'derived'/'string_human_high_confidence_edges.csv'); biogrid=pd.read_csv(EXT/'biogrid'/'derived'/'biogrid_human_edges.csv',low_memory=False); nodes=pd.read_csv(EXT/'knowledge_graph'/'derived'/'knowledge_graph_nodes.csv'); edges=pd.read_csv(EXT/'knowledge_graph'/'derived'/'knowledge_graph_edges.csv',low_memory=False)
print(react.shape,string.shape,biogrid.shape,nodes.shape,edges.shape)

(54499, 9) (230779, 5) (963018, 5) (46722, 4) (1251195, 6)


## 1. Integrity, source composition, and coverage

We check graph endpoints and count each evidence source without merging different meanings.

In [2]:
node_ids=set(nodes.node_id); audit=pd.DataFrame([{'nodes':len(nodes),'duplicate_nodes':int(nodes.node_id.duplicated().sum()),'edges':len(edges),'edges_missing_endpoint':int((~edges.source_node_id.isin(node_ids)|~edges.target_node_id.isin(node_ids)).sum()),'exact_duplicate_edges':int(edges.duplicated().sum())}]); display(audit)
display(nodes.node_type.value_counts().to_frame('nodes')); display(edges.groupby(['source_dataset','relation_type']).size().rename('edges').reset_index())
assert audit.loc[0,'edges_missing_endpoint']==0 and audit.loc[0,'exact_duplicate_edges']==0

,nodes,duplicate_nodes,edges,edges_missing_endpoint,exact_duplicate_edges
0,46722,0,1251195,0,0


,nodes
node_type,
gene,43839
pathway,2883


,source_dataset,relation_type,edges
0,BioGRID 5.0.259,curated_interaction,963018
1,Reactome,member_of_pathway,54499
2,Reactome,subpathway_of,2899
3,STRING v12.0,functional_association,230779


### What this means for the project

Every graph connection points to a known node and keeps its source. A Reactome pathway link, a STRING association, and a BioGRID experiment are shown separately because they answer different biological questions.

## 2. Reactome, STRING, and BioGRID distributions

Highly connected genes can be well studied rather than uniquely important. We inspect pathway membership, STRING score, and experimental-system frequencies.

In [3]:
gene_pathways=react.groupby(['gene_id','symbol']).pathway_id.nunique().rename('pathways').reset_index(); pathway_genes=react.groupby(['pathway_id','pathway_name']).gene_id.nunique().rename('genes').reset_index(); display(gene_pathways.nlargest(25,'pathways')); display(pathway_genes.nlargest(25,'genes'))
display(string.combined_score.describe()); display(biogrid.experimental_system.value_counts().head(30).to_frame('interaction_rows'))
degree=pd.concat([edges.source_node_id,edges.target_node_id]).value_counts().rename_axis('node_id').reset_index(name='degree').merge(nodes[['node_id','label','node_type']],on='node_id',how='left'); display(degree.loc[degree.node_type.eq('gene')].head(30))

,gene_id,symbol,pathways
151,HGNC:10417,RPS27A,223
1225,HGNC:12458,UBA52,217
1228,HGNC:12463,UBB,208
1229,HGNC:12468,UBC,204
8580,HGNC:4566,GRB2,97
395,HGNC:10845,SEM1,79
10834,HGNC:8979,PIK3R1,78
9531,HGNC:6407,KRAS,71
730,HGNC:11283,SRC,70
3123,HGNC:16889,PSMD14,70


,pathway_id,pathway_name,genes
1339,R-HSA-6798695,Neutrophil degranulation,476
2034,R-HSA-9752946,Expression and translocation of olfactory rece...,405
508,R-HSA-212436,Generic Transcription Pathway,351
2125,R-HSA-983168,Antigen processing: Ubiquitination & Proteasom...,295
820,R-HSA-418594,G alpha (i) signalling events,257
2215,R-HSA-9918481,Dengue Virus-Host Interactions,245
1627,R-HSA-8951664,Neddylation,232
1290,R-HSA-5689880,Ub-specific processing proteases,207
1431,R-HSA-72163,mRNA Splicing - Major Pathway,204
424,R-HSA-198933,Immunoregulatory interactions between a Lympho...,185


count    230779.000000
mean        862.108576
std          96.177172
min         700.000000
25%         775.000000
50%         866.000000
75%         949.000000
max         999.000000
Name: combined_score, dtype: float64

,interaction_rows
experimental_system,
Affinity Capture-MS,498384
Proximity Label-MS,152535
Two-hybrid,92100
Co-fractionation,59729
Affinity Capture-Western,44675
Reconstituted Complex,29801
Affinity Capture-RNA,20158
Cross-Linking-MS (XL-MS),16644
Negative Genetic,9439


,node_id,degree,label,node_type
0,HGNC:13870,5302,NUDT21,gene
1,HGNC:31859,4567,TRIM67,gene
2,HGNC:3236,4507,EGFR,gene
3,HGNC:7553,4487,MYC,gene
4,HGNC:11998,4251,TP53,gene
5,HGNC:18224,4018,ZRANB1,gene
6,HGNC:6407,3750,KRAS,gene
7,HGNC:10289,3577,RPA1,gene
8,HGNC:2553,3429,CUL3,gene
9,HGNC:10290,3330,RPA2,gene


### What this means for the project

A hub gene can connect to many genes because it is central biology or because researchers studied it often. Degree and source counts are explanation context, not automatic confidence in a queried pair.

## 3. Example two-gene explanations

For each pair we show direct STRING/BioGRID evidence and shared Reactome pathways. Absence means no record in these versions, not proof that no relationship exists.

In [4]:
gene_nodes=nodes.loc[nodes.node_type.eq('gene')].drop_duplicates('label').set_index('label').node_id.to_dict()
def explain(a,b):
    ga,gb=gene_nodes.get(a),gene_nodes.get(b)
    if not ga or not gb:return {'pair':f'{a}-{b}','resolved':False}
    x,y=sorted([ga,gb]); sp=string.loc[(string.gene_a_id.eq(x)&string.gene_b_id.eq(y))]; bg=biogrid.loc[(biogrid.gene_a_id.eq(x)&biogrid.gene_b_id.eq(y))]; pa=set(react.loc[react.gene_id.eq(ga),'pathway_id']); pb=set(react.loc[react.gene_id.eq(gb),'pathway_id']); shared=react.loc[react.pathway_id.isin(pa&pb),['pathway_id','pathway_name']].drop_duplicates()
    return {'pair':f'{a}-{b}','resolved':True,'STRING_edges':len(sp),'max_STRING_score':sp.combined_score.max() if len(sp) else np.nan,'BioGRID_systems':bg.experimental_system.nunique(),'BioGRID_records':bg.n_curated_records.sum(),'shared_Reactome_pathways':len(shared),'example_shared_pathways':' | '.join(shared.pathway_name.head(5))}
display(pd.DataFrame([explain(*p) for p in [('BRCA1','BRCA2'),('ARID1A','EZH2'),('EML4','ALK'),('EGFR','KRAS')]]))

,pair,resolved,STRING_edges,max_STRING_score,BioGRID_systems,BioGRID_records,shared_Reactome_pathways,example_shared_pathways
0,BRCA1-BRCA2,True,1,999,7,16,11,Resolution of D-loop Structures through Hollid...
1,ARID1A-EZH2,True,1,877,3,5,0,
2,EML4-ALK,True,1,999,0,0,2,Signaling by ALK fusions and activated point m...
3,EGFR-KRAS,True,1,997,3,5,11,RAF/MAP kinase cascade | SHC1 events in ERBB2 ...


### What this means for the project

The dashboard can say, for example, that two genes share named pathways, have a STRING association, or were connected by a particular BioGRID experiment. Those statements explain known biology; they do not replace the cell-line mutation and CRISPR evidence.

## Decisions carried into cross-layer EDA

- Keep every network source and evidence type visible.
- Use shared pathways, STRING association, and BioGRID experiments as explanations, not labels.
- Resolve query genes through canonical project gene IDs.
- Report database/version absence as "no record found," not biological absence.
- Avoid hub degree as a hidden confidence score.